# TP3 Chatbots basados en recuperación de la información


# Alumno: Moises Lobayza

# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

# Librerías

In [ ]:
!pip install spacy --quiet
!python -m spacy download es_core_news_sm --quiet
!pip install sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 71.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import re
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# Actividades

### 1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. ( 3 puntos )

1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

**Respuesta:**

## Aplicación elegida

La aplicación elegida es un chatbot para un **consultorio médico privado**.

El **objetivo** del chatbot es responder consultas frecuentes de pacientes relacionadas con turnos, horarios de atención, ubicación del consultorio, formas de pago, cancelaciones, reprogramaciones, estudios previos, recetas, certificados y consultas administrativas generales.

El chatbot no reemplaza la atención médica profesional ni realiza diagnósticos. **Su función principal es brindar información administrativa y orientar al paciente para que pueda comunicarse correctamente con el consultorio o solicitar un turno**.

In [ ]:
data = [
    {
        "pregunta": "¿Cómo puedo sacar un turno?",
        "respuesta": "Podés solicitar un turno escribiendo por WhatsApp al consultorio e indicando tu nombre, apellido, obra social y motivo de consulta."
    },
    {
        "pregunta": "¿Cuáles son los horarios de atención?",
        "respuesta": "El consultorio atiende de lunes a viernes de 9:00 a 18:00 hs. Los horarios pueden variar según la disponibilidad del profesional."
    },
    {
        "pregunta": "¿Dónde queda el consultorio?",
        "respuesta": "El consultorio se encuentra en Córdoba Capital. La dirección exacta se informa al confirmar el turno."
    },
    {
        "pregunta": "¿Atienden por obra social?",
        "respuesta": "Sí, el consultorio trabaja con algunas obras sociales. Para confirmar cobertura, indicá cuál es tu obra social al solicitar el turno."
    },
    {
        "pregunta": "¿Puedo atenderme de forma particular?",
        "respuesta": "Sí, también se atienden pacientes particulares. El valor de la consulta se informa al momento de solicitar el turno."
    },
    {
        "pregunta": "¿Cómo hago para cancelar un turno?",
        "respuesta": "Para cancelar un turno, escribí al WhatsApp del consultorio con tu nombre, apellido y fecha del turno que querés cancelar."
    },
    {
        "pregunta": "¿Puedo reprogramar mi turno?",
        "respuesta": "Sí, podés reprogramar tu turno escribiendo al consultorio. Se te ofrecerán nuevas opciones según la disponibilidad."
    },
    {
        "pregunta": "¿Qué datos necesito para pedir un turno?",
        "respuesta": "Para pedir un turno necesitás informar nombre, apellido, DNI, obra social si tenés, motivo de consulta y disponibilidad horaria."
    },
    {
        "pregunta": "¿Atienden urgencias?",
        "respuesta": "El consultorio no atiende urgencias. En caso de emergencia médica, dirigite a una guardia o llamá al servicio de emergencias correspondiente."
    },
    {
        "pregunta": "¿Puedo pedir una receta por WhatsApp?",
        "respuesta": "Las recetas deben ser evaluadas por el profesional. En algunos casos pueden solicitarse por WhatsApp si ya sos paciente del consultorio."
    },
    {
        "pregunta": "¿Hacen certificados médicos?",
        "respuesta": "Sí, el profesional puede realizar certificados médicos cuando corresponda, luego de una evaluación en consulta."
    },
    {
        "pregunta": "¿Cuánto dura una consulta?",
        "respuesta": "La duración aproximada de una consulta es de 20 a 30 minutos, aunque puede variar según el motivo de atención."
    },
    {
        "pregunta": "¿Qué pasa si llego tarde al turno?",
        "respuesta": "Si llegás tarde, el profesional intentará atenderte según la disponibilidad del día. En algunos casos puede ser necesario reprogramar."
    },
    {
        "pregunta": "¿Debo llevar estudios anteriores?",
        "respuesta": "Sí, si tenés estudios previos, análisis, recetas o informes médicos, es recomendable llevarlos a la consulta."
    },
    {
        "pregunta": "¿Puedo ir acompañado a la consulta?",
        "respuesta": "Sí, podés asistir acompañado si lo necesitás. En algunos casos el ingreso del acompañante puede depender del espacio disponible."
    },
    {
        "pregunta": "¿Atienden niños?",
        "respuesta": "La atención a niños depende de la especialidad del profesional. Consultá previamente indicando la edad del paciente."
    },
    {
        "pregunta": "¿Cuáles son las formas de pago?",
        "respuesta": "El consultorio acepta efectivo y transferencias. Otros medios de pago pueden consultarse al momento de confirmar el turno."
    },
    {
        "pregunta": "¿Entregan factura?",
        "respuesta": "Sí, el consultorio puede emitir comprobante o factura por la atención realizada."
    },
    {
        "pregunta": "¿Puedo consultar resultados de estudios por WhatsApp?",
        "respuesta": "Los resultados deben ser evaluados por el profesional. Podés enviarlos por WhatsApp, pero la interpretación se realizará en consulta o según indicación médica."
    },
    {
        "pregunta": "¿Cómo confirmo mi turno?",
        "respuesta": "Para confirmar tu turno, respondé el mensaje de confirmación enviado por el consultorio indicando que vas a asistir."
    }
]

df = pd.DataFrame(data)

df

,pregunta,respuesta
0,¿Cómo puedo sacar un turno?,Podés solicitar un turno escribiendo por Whats...
1,¿Cuáles son los horarios de atención?,El consultorio atiende de lunes a viernes de 9...
2,¿Dónde queda el consultorio?,El consultorio se encuentra en Córdoba Capital...
3,¿Atienden por obra social?,"Sí, el consultorio trabaja con algunas obras s..."
4,¿Puedo atenderme de forma particular?,"Sí, también se atienden pacientes particulares..."
5,¿Cómo hago para cancelar un turno?,"Para cancelar un turno, escribí al WhatsApp de..."
6,¿Puedo reprogramar mi turno?,"Sí, podés reprogramar tu turno escribiendo al ..."
7,¿Qué datos necesito para pedir un turno?,"Para pedir un turno necesitás informar nombre,..."
8,¿Atienden urgencias?,El consultorio no atiende urgencias. En caso d...
9,¿Puedo pedir una receta por WhatsApp?,Las recetas deben ser evaluadas por el profesi...


# Limpieza general antes de aplicar los modelos.

Para simplificar el trabajo se utilizó el mismo preprocesamiento en ambos modelos. Sin embargo, **TF-IDF depende más de la limpieza porque trabaja con palabras y frecuencias, mientras que los embeddings pueden trabajar mejor con texto natural y no siempre requieren una limpieza tan fuerte.**

No se eliminaron stop words porque el dataset contiene preguntas cortas y administrativas, donde algunas palabras funcionales pueden aportar contexto. Se mantuvo una limpieza básica para estandarizar el texto sin perder información relevante.


In [ ]:
def limpiar_texto(texto):
    texto = str(texto).lower()

    # Quitar tildes
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

    # Quitar signos de puntuación y caracteres especiales
    texto = re.sub(r'[^a-zñ\s]', '', texto)

    # Quitar espacios extra
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto


# Aplicar limpieza a preguntas y respuestas
df["pregunta_limpia"] = df["pregunta"].apply(limpiar_texto)
df["respuesta_limpia"] = df["respuesta"].apply(limpiar_texto)

# Verificar resultado
df[["pregunta", "pregunta_limpia", "respuesta", "respuesta_limpia"]].head()

,pregunta,pregunta_limpia,respuesta,respuesta_limpia
0,¿Cómo puedo sacar un turno?,como puedo sacar un turno,Podés solicitar un turno escribiendo por Whats...,podes solicitar un turno escribiendo por whats...
1,¿Cuáles son los horarios de atención?,cuales son los horarios de atencion,El consultorio atiende de lunes a viernes de 9...,el consultorio atiende de lunes a viernes de a...
2,¿Dónde queda el consultorio?,donde queda el consultorio,El consultorio se encuentra en Córdoba Capital...,el consultorio se encuentra en cordoba capital...
3,¿Atienden por obra social?,atienden por obra social,"Sí, el consultorio trabaja con algunas obras s...",si el consultorio trabaja con algunas obras so...
4,¿Puedo atenderme de forma particular?,puedo atenderme de forma particular,"Sí, también se atienden pacientes particulares...",si tambien se atienden pacientes particulares ...


# Funciones de uso común

En esta sección se definieron funciones generales que pueden ser reutilizadas tanto por el chatbot basado en **TF-IDF** como por el chatbot basado en **embeddings**.

**La función `chatbot_retrieval()` contiene la lógica principal del chatbot.** Recibe una consulta del usuario, la vectoriza con el método correspondiente, calcula la similitud del coseno contra las preguntas del dataset y devuelve la respuesta asociada a la pregunta más similar. También aplica un umbral mínimo de similitud para evitar responder cuando no encuentra una coincidencia adecuada.

**La función `preparar_matriz_preguntas()` se utiliza para convertir las preguntas limpias del dataset en una matriz de vectores.** Esta función es general porque puede recibir distintos métodos de vectorización, por ejemplo TF-IDF o embeddings.

In [ ]:
def chatbot_retrieval(consulta_usuario, matriz_referencia, funcion_vectorizar, umbral):
    # Vectorizar la consulta según el modelo elegido
    consulta_vectorizada = funcion_vectorizar(consulta_usuario)

    # Calcular similitud del coseno
    similitudes = cosine_similarity(consulta_vectorizada, matriz_referencia)

    # Buscar la pregunta más similar
    indice_mas_similar = similitudes.argmax()

    # Obtener mejor similitud
    mejor_similitud = similitudes[0, indice_mas_similar]

    # Aplicar umbral mínimo
    if mejor_similitud < umbral:
        return "No encontré una respuesta adecuada. Te recomiendo comunicarte directamente con el consultorio."

    # Devolver respuesta original asociada
    return df.iloc[indice_mas_similar]["respuesta"]

In [ ]:
def preparar_matriz_preguntas(preguntas_limpias, funcion_vectorizar_dataset):
    return funcion_vectorizar_dataset(preguntas_limpias)

# 2) Crear el chatbot utilizando TFIDF y similitud del coseno. (1 punto)

**En este bloque se construye el chatbot basado en TF-IDF.**

Primero se crea el vectorizador `TfidfVectorizer`, que permite transformar texto en vectores numéricos según la importancia de las palabras.

Luego, la función `vectorizar_dataset_tfidf()` aplica `fit_transform()` sobre las preguntas limpias del dataset. Esto significa que el vectorizador aprende el vocabulario de las preguntas y genera la matriz `matriz_tfidf`, que contiene los vectores de referencia.

Después, la función `vectorizar_tfidf()` se utiliza para transformar una **consulta** nueva del usuario usando el mismo vectorizador ya ajustado.

Finalmente, `chatbot_tfidf()` llama a la función general `chatbot_retrieval()`, pasándole la matriz TF-IDF, la función de vectorización correspondiente y un umbral mínimo de similitud de 0.2.

In [ ]:
# Crear vectorizador TF-IDF
vectorizador_tfidf = TfidfVectorizer()

def vectorizar_dataset_tfidf(preguntas_limpias):
    return vectorizador_tfidf.fit_transform(preguntas_limpias)

matriz_tfidf = preparar_matriz_preguntas(
    df["pregunta_limpia"],
    vectorizar_dataset_tfidf
)

In [ ]:
def vectorizar_tfidf(consulta_usuario):
    consulta_limpia = limpiar_texto(consulta_usuario)
    return vectorizador_tfidf.transform([consulta_limpia])

In [ ]:
def chatbot_tfidf(consulta_usuario):
    return chatbot_retrieval(
        consulta_usuario,
        matriz_tfidf,
        vectorizar_tfidf,
        umbral=0.2
    )

# 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado eligió.

**En este bloque se construye el chatbot basado en embeddings.**

Primero se carga el modelo preentrenado `paraphrase-multilingual-MiniLM-L12-v2`, que permite **transformar oraciones completas en vectores semánticos. Se eligió este modelo porque es multilingüe, funciona con español y está pensado para comparar similitud entre frases**.

Luego, la función `vectorizar_dataset_embeddings()` convierte las preguntas limpias del dataset en embeddings. Esos vectores se guardan en `embeddings_preguntas`, que funciona como matriz de referencia.

Después, la función `vectorizar_embeddings()` transforma cada consulta nueva del usuario en un embedding usando el mismo modelo.

Finalmente, `chatbot_embeddings()` llama a la función general `chatbot_retrieval()`, pasándole los embeddings de las preguntas, la función de vectorización correspondiente y un umbral mínimo de similitud de 0.45.

In [ ]:
modelo_embeddings = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def vectorizar_dataset_embeddings(preguntas_limpias):
    return modelo_embeddings.encode(preguntas_limpias.tolist())

embeddings_preguntas = preparar_matriz_preguntas(
    df["pregunta_limpia"],
    vectorizar_dataset_embeddings
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def vectorizar_embeddings(consulta_usuario):
    consulta_limpia = limpiar_texto(consulta_usuario)
    return modelo_embeddings.encode([consulta_limpia])

In [ ]:
def chatbot_embeddings(consulta_usuario):
    return chatbot_retrieval(
        consulta_usuario,
        embeddings_preguntas,
        vectorizar_embeddings,
        umbral=0.45
    )

# 4) Muestra ambos chatbots funcionando (1 punto)
Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

# Consultas TF-IDF.

In [ ]:
print(chatbot_tfidf("quiero sacar un turno"))
print(chatbot_tfidf("donde queda el consultorio"))
print(chatbot_tfidf("donde queda el consultorio?????"))
print(chatbot_tfidf("Hola, atienden con obra social???"))
print(chatbot_tfidf("necesito cancelar mi cita"))
print(chatbot_tfidf("tengo una urgencia medica"))
print(chatbot_tfidf("ajsjhshshhshshs"))
print(chatbot_tfidf("puedo pagar por transferencia"))
print(chatbot_tfidf("tengo que llevar estudios anteriores"))
print(chatbot_tfidf("puedo pedir una receta por whatsapp"))

Podés solicitar un turno escribiendo por WhatsApp al consultorio e indicando tu nombre, apellido, obra social y motivo de consulta.
El consultorio se encuentra en Córdoba Capital. La dirección exacta se informa al confirmar el turno.
El consultorio se encuentra en Córdoba Capital. La dirección exacta se informa al confirmar el turno.
Sí, el consultorio trabaja con algunas obras sociales. Para confirmar cobertura, indicá cuál es tu obra social al solicitar el turno.
Sí, podés reprogramar tu turno escribiendo al consultorio. Se te ofrecerán nuevas opciones según la disponibilidad.
La duración aproximada de una consulta es de 20 a 30 minutos, aunque puede variar según el motivo de atención.
No encontré una respuesta adecuada. Te recomiendo comunicarte directamente con el consultorio.
Las recetas deben ser evaluadas por el profesional. En algunos casos pueden solicitarse por WhatsApp si ya sos paciente del consultorio.
Sí, si tenés estudios previos, análisis, recetas o informes médicos, es

# Análisis de las respuestas del modelo TF-IDF.

In [ ]:
consultas_prueba = [
    "quiero sacar un turno",
    "donde queda el consultorio",
    "donde queda el consultorio?????",
    "Hola, atienden con obra social???",
    "necesito cancelar mi cita",
    "tengo una urgencia medica",
    "ajsjhshshhshshs",
    "puedo pagar por transferencia",
    "tengo que llevar estudios anteriores",
    "puedo pedir una receta por whatsapp"
]

respuestas_tfidf = [chatbot_tfidf(consulta) for consulta in consultas_prueba]

evaluacion_tfidf = [
    "Correcta",
    "Correcta",
    "Correcta",
    "Correcta",
    "Incorrecta",
    "Incorrecta",
    "Correcta",
    "Incorrecta",
    "Correcta",
    "Correcta"
]

observaciones_tfidf = [
    "Detectó correctamente la intención de solicitar turno.",
    "Respondió correctamente sobre la ubicación.",
    "La limpieza permitió ignorar los signos de puntuación.",
    "Detectó correctamente la consulta sobre obra social.",
    "Confundió cancelar con reprogramar.",
    "No detectó correctamente que era una urgencia.",
    "Correctamente no encontró una respuesta adecuada.",
    "Confundió la consulta de pago con recetas.",
    "Respondió correctamente sobre estudios previos.",
    "Respondió correctamente sobre recetas por WhatsApp."
]

df_resultados_tfidf = pd.DataFrame({
    "Consulta": consultas_prueba,
    "Respuesta_tfidf": respuestas_tfidf,
    "Evaluacion": evaluacion_tfidf,
    "Observacion": observaciones_tfidf
})

df_resultados_tfidf

,Consulta,Respuesta_tfidf,Evaluacion,Observacion
0,quiero sacar un turno,Podés solicitar un turno escribiendo por Whats...,Correcta,Detectó correctamente la intención de solicita...
1,donde queda el consultorio,El consultorio se encuentra en Córdoba Capital...,Correcta,Respondió correctamente sobre la ubicación.
2,donde queda el consultorio?????,El consultorio se encuentra en Córdoba Capital...,Correcta,La limpieza permitió ignorar los signos de pun...
3,"Hola, atienden con obra social???","Sí, el consultorio trabaja con algunas obras s...",Correcta,Detectó correctamente la consulta sobre obra s...
4,necesito cancelar mi cita,"Sí, podés reprogramar tu turno escribiendo al ...",Incorrecta,Confundió cancelar con reprogramar.
5,tengo una urgencia medica,La duración aproximada de una consulta es de 2...,Incorrecta,No detectó correctamente que era una urgencia.
6,ajsjhshshhshshs,No encontré una respuesta adecuada. Te recomie...,Correcta,Correctamente no encontró una respuesta adecuada.
7,puedo pagar por transferencia,Las recetas deben ser evaluadas por el profesi...,Incorrecta,Confundió la consulta de pago con recetas.
8,tengo que llevar estudios anteriores,"Sí, si tenés estudios previos, análisis, recet...",Correcta,Respondió correctamente sobre estudios previos.
9,puedo pedir una receta por whatsapp,Las recetas deben ser evaluadas por el profesi...,Correcta,Respondió correctamente sobre recetas por What...


# Conteo final modelo TF-IDF

In [ ]:
df_resultados_tfidf["Evaluacion"].value_counts()

,count
Evaluacion,
Correcta,7
Incorrecta,3


# Conclusión del modelo TF-IDF

El chatbot basado en TF-IDF respondió correctamente en 7 de 10 consultas. Funcionó bien cuando la pregunta del usuario tenía palabras similares a las preguntas del dataset, como en los casos de turnos, ubicación, obra social, estudios anteriores y recetas.

Sin embargo, presentó **errores** cuando el usuario usó palabras distintas o sinónimos, por ejemplo “cita” en lugar de “turno”, o cuando la consulta requería entender mejor el significado. **Esto muestra que TF-IDF depende mucho de la coincidencia de palabras y tiene limitaciones para interpretar el contexto semántico.**

# Consultas Embeddings.

In [ ]:
print(chatbot_embeddings("quiero sacar un turno"))
print(chatbot_embeddings("donde queda el consultorio"))
print(chatbot_embeddings("donde queda el consultorio?????"))
print(chatbot_embeddings("Hola, atienden con obra social???"))
print(chatbot_embeddings("necesito cancelar mi cita"))
print(chatbot_embeddings("tengo una urgencia medica"))
print(chatbot_embeddings("ajsjhshshhshshs"))
print(chatbot_embeddings("puedo pagar por transferencia"))
print(chatbot_embeddings("tengo que llevar estudios anteriores"))
print(chatbot_embeddings("puedo pedir una receta por whatsapp"))

Podés solicitar un turno escribiendo por WhatsApp al consultorio e indicando tu nombre, apellido, obra social y motivo de consulta.
El consultorio se encuentra en Córdoba Capital. La dirección exacta se informa al confirmar el turno.
El consultorio se encuentra en Córdoba Capital. La dirección exacta se informa al confirmar el turno.
Sí, el consultorio trabaja con algunas obras sociales. Para confirmar cobertura, indicá cuál es tu obra social al solicitar el turno.
Para cancelar un turno, escribí al WhatsApp del consultorio con tu nombre, apellido y fecha del turno que querés cancelar.
El consultorio no atiende urgencias. En caso de emergencia médica, dirigite a una guardia o llamá al servicio de emergencias correspondiente.
La atención a niños depende de la especialidad del profesional. Consultá previamente indicando la edad del paciente.
Sí, el consultorio puede emitir comprobante o factura por la atención realizada.
Sí, si tenés estudios previos, análisis, recetas o informes médicos

# Análisis de las respuestas del modelo Embeddings.

In [ ]:
respuestas_embeddings = [chatbot_embeddings(consulta) for consulta in consultas_prueba]

evaluacion_embeddings = [
    "Correcta",
    "Correcta",
    "Correcta",
    "Correcta",
    "Correcta",
    "Correcta",
    "Incorrecta",
    "Incorrecta",
    "Correcta",
    "Correcta"
]

observaciones_embeddings = [
    "Detectó correctamente la intención de solicitar turno.",
    "Respondió correctamente sobre la ubicación.",
    "La limpieza permitió ignorar los signos de puntuación.",
    "Detectó correctamente la consulta sobre obra social.",
    "Detectó correctamente la intención de cancelar un turno.",
    "Respondió correctamente ante una urgencia médica.",
    "No rechazó la consulta sin sentido y devolvió una respuesta incorrecta.",
    "Confundió la consulta sobre forma de pago con emisión de factura.",
    "Respondió correctamente sobre estudios previos.",
    "Respondió correctamente sobre recetas por WhatsApp."
]

df_resultados_embeddings = pd.DataFrame({
    "consulta": consultas_prueba,
    "respuesta_embeddings": respuestas_embeddings,
    "evaluacion": evaluacion_embeddings,
    "observacion": observaciones_embeddings
})

df_resultados_embeddings

,consulta,respuesta_embeddings,evaluacion,observacion
0,quiero sacar un turno,Podés solicitar un turno escribiendo por Whats...,Correcta,Detectó correctamente la intención de solicita...
1,donde queda el consultorio,El consultorio se encuentra en Córdoba Capital...,Correcta,Respondió correctamente sobre la ubicación.
2,donde queda el consultorio?????,El consultorio se encuentra en Córdoba Capital...,Correcta,La limpieza permitió ignorar los signos de pun...
3,"Hola, atienden con obra social???","Sí, el consultorio trabaja con algunas obras s...",Correcta,Detectó correctamente la consulta sobre obra s...
4,necesito cancelar mi cita,"Para cancelar un turno, escribí al WhatsApp de...",Correcta,Detectó correctamente la intención de cancelar...
5,tengo una urgencia medica,El consultorio no atiende urgencias. En caso d...,Correcta,Respondió correctamente ante una urgencia médica.
6,ajsjhshshhshshs,La atención a niños depende de la especialidad...,Incorrecta,No rechazó la consulta sin sentido y devolvió ...
7,puedo pagar por transferencia,"Sí, el consultorio puede emitir comprobante o ...",Incorrecta,Confundió la consulta sobre forma de pago con ...
8,tengo que llevar estudios anteriores,"Sí, si tenés estudios previos, análisis, recet...",Correcta,Respondió correctamente sobre estudios previos.
9,puedo pedir una receta por whatsapp,Las recetas deben ser evaluadas por el profesi...,Correcta,Respondió correctamente sobre recetas por What...


# Conteo final modelo Embeddings.

In [ ]:
df_resultados_embeddings["evaluacion"].value_counts()

,count
evaluacion,
Correcta,8
Incorrecta,2


### Conclusión del modelo con embeddings

El chatbot basado en embeddings respondió correctamente en 8 de 10 consultas. En general tuvo mejor desempeño que TF-IDF porque logró interpretar mejor consultas formuladas con palabras distintas, como “necesito cancelar mi cita” o “tengo una urgencia médica”.

De todos modos, también cometió errores: ante una consulta sin sentido devolvió una respuesta incorrecta, y confundió una pregunta sobre transferencia con una respuesta sobre factura. **Esto muestra que los embeddings capturan mejor el significado, pero también necesitan un buen umbral de similitud y una base de respuestas bien diseñada.**

# 5) Añade tus conclusiones de todo lo realizado (2 punto)
Resalte e indique en cuáles respuestas falla o tiene problemas.

Cuáles preguntas confunde.

Compare los resultados de los chatbots.

## 5) Conclusiones finales

En este trabajo se construyeron dos chatbots de recuperación de información para un consultorio médico privado: uno con TF-IDF y otro con embeddings. Ambos usaron limpieza de texto y similitud del coseno para buscar la pregunta más parecida del dataset.

El chatbot con TF-IDF respondió correctamente 7 de 10 consultas. Funcionó bien cuando había coincidencia directa de palabras, pero falló con sinónimos o frases menos similares. Por ejemplo, confundió “cancelar mi cita” con reprogramar, “urgencia médica” con duración de consulta y “pagar por transferencia” con recetas.

El chatbot con embeddings respondió correctamente 8 de 10 consultas. Tuvo mejor desempeño porque interpretó mejor el significado de frases distintas, como “cancelar mi cita” y “urgencia médica”. Sin embargo, también falló ante una consulta sin sentido y confundió la pregunta sobre transferencia con factura.

En comparación, TF-IDF es más simple y fácil de interpretar, pero depende mucho de las palabras exactas. **Los embeddings resultaron más adecuados para este caso, aunque también requieren ajustar el umbral de similitud y mejorar la base de respuestas.**

## Referencias

- Scikit-learn. Documentación oficial de `TfidfVectorizer`. Se utilizó como referencia para la vectorización TF-IDF de las preguntas del dataset.

- Scikit-learn. Documentación oficial de `cosine_similarity`. Se utilizó como referencia para calcular la similitud del coseno entre la consulta del usuario y las preguntas vectorizadas.

- Sentence Transformers. Documentación sobre modelos preentrenados. Se utilizó como referencia para el uso de modelos de embeddings preentrenados.

- Hugging Face. Modelo `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`. Se utilizó como embedding preentrenado para representar oraciones completas y comparar similitud semántica.

- Apuntes y notebook de clase sobre procesamiento de lenguaje natural, TF-IDF, embeddings y similitud entre textos.

- LLM usados para consulta: Gpt, Gemini.